In [141]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg


# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 데이터 로드

In [142]:
df_c = pd.read_csv("../../tft_game_dataset/TFT_Challenger_MatchData.csv")
df_g = pd.read_csv("../../tft_game_dataset/TFT_GrandMaster_MatchData.csv")
df_m = pd.read_csv("../../tft_game_dataset/TFT_Master_MatchData.csv")
df_d = pd.read_csv("../../tft_game_dataset/TFT_Diamond_MatchData.csv")
df_p = pd.read_csv("../../tft_game_dataset/TFT_Platinum_MatchData.csv")

df_champion = pd.read_csv("../../tft_game_dataset/TFT_Champion_CurrentVersion.csv")
df_item = pd.read_csv("../../tft_game_dataset/TFT_Item_CurrentVersion.csv")

In [143]:
df_c.head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion
0,KR_4247538593,2142.470703,8,35,1,2134.272217,"{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'S...","{'JarvanIV': {'items': [27], 'star': 3}, 'Sona..."
1,KR_4247538593,2142.470703,9,35,2,2134.272217,"{'Blaster': 2, 'Mercenary': 1, 'Rebel': 6, 'Se...","{'Malphite': {'items': [7], 'star': 2}, 'Yasuo..."
2,KR_4247538593,2142.470703,8,34,3,2073.459229,"{'Cybernetic': 1, 'DarkStar': 3, 'Demolitionis...","{'KaiSa': {'items': [99, 2, 23], 'star': 2}, '..."
3,KR_4247538593,2142.470703,8,33,4,1998.146729,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 1,...","{'KaiSa': {'items': [44, 37], 'star': 2}, 'Ann..."
4,KR_4247538593,2142.470703,9,33,5,1986.443237,"{'Blaster': 2, 'Demolitionist': 2, 'Mercenary'...","{'Ziggs': {'items': [], 'star': 1}, 'Yasuo': {..."


### 각 테이블에 티어 컬럼 추가

In [144]:
df_c["tier"] = "Challenger"
df_g["tier"] = "GrandMaster"
df_m["tier"] = "Master"
df_d["tier"] = "Diamond"
df_p["tier"] = "Platinum"

## 매치데이터 머지

In [145]:
merged_df = pd.concat([df_c, df_g, df_m, df_d, df_p], ignore_index=True)

In [146]:
merged_df.head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4247538593,2142.470703,8,35,1,2134.272217,"{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'S...","{'JarvanIV': {'items': [27], 'star': 3}, 'Sona...",Challenger
1,KR_4247538593,2142.470703,9,35,2,2134.272217,"{'Blaster': 2, 'Mercenary': 1, 'Rebel': 6, 'Se...","{'Malphite': {'items': [7], 'star': 2}, 'Yasuo...",Challenger
2,KR_4247538593,2142.470703,8,34,3,2073.459229,"{'Cybernetic': 1, 'DarkStar': 3, 'Demolitionis...","{'KaiSa': {'items': [99, 2, 23], 'star': 2}, '...",Challenger
3,KR_4247538593,2142.470703,8,33,4,1998.146729,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 1,...","{'KaiSa': {'items': [44, 37], 'star': 2}, 'Ann...",Challenger
4,KR_4247538593,2142.470703,9,33,5,1986.443237,"{'Blaster': 2, 'Demolitionist': 2, 'Mercenary'...","{'Ziggs': {'items': [], 'star': 1}, 'Yasuo': {...",Challenger


In [147]:
print(merged_df["tier"].value_counts())

tier
GrandMaster    80000
Diamond        80000
Platinum       80000
Challenger     79999
Master         79999
Name: count, dtype: int64


# 기본 구조 확인

In [148]:
print(merged_df.shape)
print(merged_df.columns)
print(merged_df.info())

(399998, 9)
Index(['gameId', 'gameDuration', 'level', 'lastRound', 'Ranked',
       'ingameDuration', 'combination', 'champion', 'tier'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 399998 entries, 0 to 399997
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   gameId          399998 non-null  str    
 1   gameDuration    399998 non-null  float64
 2   level           399998 non-null  int64  
 3   lastRound       399998 non-null  int64  
 4   Ranked          399998 non-null  int64  
 5   ingameDuration  399998 non-null  float64
 6   combination     399998 non-null  str    
 7   champion        399998 non-null  str    
 8   tier            399998 non-null  str    
dtypes: float64(2), int64(3), str(4)
memory usage: 27.5 MB
None


In [149]:
merged_df.head()
merged_df.sample(3)

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
242139,KR_4228629140,2155.334229,8,31,6,1771.401978,"{'Blaster': 1, 'Chrono': 4, 'Cybernetic': 1, '...","{'TwistedFate': {'items': [], 'star': 2}, 'Cai...",Diamond
225458,KR_4359449493,2193.551270,8,34,5,1843.657959,"{'Chrono': 2, 'Cybernetic': 1, 'Infiltrator': ...","{'TwistedFate': {'items': [], 'star': 2}, 'Mal...",Master
10463,KR_4300292230,2172.491943,5,23,8,1242.825928,"{'DarkStar': 1, 'Protector': 4, 'Rebel': 1, 'S...","{'JarvanIV': {'items': [], 'star': 3}, 'Sona':...",Challenger


중복행 검색

In [150]:
merged_df.isnull().sum()
merged_df.duplicated().sum()

np.int64(68)

In [151]:
merged_df[merged_df.duplicated()].head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
100912,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100913,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100914,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100915,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100916,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster


In [152]:
merged_df["tier"].value_counts()

tier
GrandMaster    80000
Diamond        80000
Platinum       80000
Challenger     79999
Master         79999
Name: count, dtype: int64

In [153]:
merged_df["Ranked"].value_counts().sort_index()

Ranked
0       92
1    49986
2    49985
3    49988
4    49988
5    49988
6    49987
7    49988
8    49996
Name: count, dtype: int64

등수가 0인건 뭐지??

In [154]:
merged_df[merged_df["Ranked"] == 0].head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
100911,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100912,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100913,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100914,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster
100915,KR_4336328756,125.278297,3,4,0,125.278297,{},{},GrandMaster


In [155]:
#Ranked 0인거 몇개?
merged_df[merged_df["Ranked"] == 0].shape

(92, 9)

In [156]:
#Ranked 0인거 티어별로 카운트
merged_df[merged_df["Ranked"] == 0]["tier"].value_counts()

tier
Platinum       71
Master         14
GrandMaster     7
Name: count, dtype: int64

- 등수는 1~8등으로 이루어져있기 때문에 0등은 사실상 말이 안돼는 숫자. 
- gameId를 기준으로 각 게임당 8명이 이루어져있는 것인지 확인 필요.

gameId별 행 개수 확인

In [157]:
#8명이 아닌 게임만 따로 보기
game_counts = merged_df.groupby("gameId").size()

bad_games_count = game_counts[game_counts != 8]
bad_games_count

gameId
KR_4263820118    16
KR_4313697214    16
KR_4318806255     7
KR_4320079059    16
KR_4335870255     7
KR_4344513862    16
KR_4347884427    16
KR_4357966612    16
KR_4358922415    16
KR_4361594426    16
KR_4361773461    16
KR_4362846604    16
KR_4364453473    16
KR_4365284161    16
KR_4378896137    16
KR_4381231118    16
KR_4387025966    16
KR_4387247645    16
KR_4388134708    16
KR_4393787119    16
KR_4398618214    16
dtype: int64

In [158]:
merged_df[merged_df["gameId"] == "KR_4263820118"].sort_values("Ranked")

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
160719,KR_4263820118,2302.919189,9,38,1,2294.606934,"{'Chrono': 2, 'DarkStar': 3, 'ManaReaver': 1, ...","{'Poppy': {'items': [], 'star': 2}, 'Mordekais...",Master
329665,KR_4263820118,2302.919189,9,38,1,2294.606934,"{'Chrono': 2, 'DarkStar': 3, 'ManaReaver': 1, ...","{'Poppy': {'items': [], 'star': 2}, 'Mordekais...",Platinum
329663,KR_4263820118,2302.919189,8,38,2,2294.606934,"{'Cybernetic': 1, 'DarkStar': 3, 'Demolitionis...","{'JarvanIV': {'items': [], 'star': 1}, 'KaiSa'...",Platinum
160720,KR_4263820118,2302.919189,8,38,2,2294.606934,"{'Cybernetic': 1, 'DarkStar': 3, 'Demolitionis...","{'JarvanIV': {'items': [], 'star': 1}, 'KaiSa'...",Master
160721,KR_4263820118,2302.919189,8,37,3,2223.663086,"{'Blaster': 4, 'Chrono': 2, 'Cybernetic': 2, '...","{'Malphite': {'items': [], 'star': 2}, 'Blitzc...",Master
329664,KR_4263820118,2302.919189,8,37,3,2223.663086,"{'Blaster': 4, 'Chrono': 2, 'Cybernetic': 2, '...","{'Malphite': {'items': [], 'star': 2}, 'Blitzc...",Platinum
329662,KR_4263820118,2302.919189,8,33,4,1990.959717,"{'Chrono': 1, 'Cybernetic': 1, 'DarkStar': 3, ...","{'Leona': {'items': [], 'star': 2}, 'Mordekais...",Platinum
160722,KR_4263820118,2302.919189,8,33,4,1990.959717,"{'Chrono': 1, 'Cybernetic': 1, 'DarkStar': 3, ...","{'Leona': {'items': [], 'star': 2}, 'Mordekais...",Master
329667,KR_4263820118,2302.919189,8,33,5,1982.455444,"{'Blaster': 2, 'Demolitionist': 1, 'Mercenary'...","{'Ziggs': {'items': [], 'star': 2}, 'Yasuo': {...",Platinum
160723,KR_4263820118,2302.919189,8,33,5,1982.455444,"{'Blaster': 2, 'Demolitionist': 1, 'Mercenary'...","{'Ziggs': {'items': [], 'star': 2}, 'Yasuo': {...",Master


- 각 csv테이블은 해당 티어의 게임만 들어있는 것이 아니고 마스터 유저기준으로 수집한 매치 데이터, 플래티넘 유저 기준으로 수집한 데이터처럼 만들어진 것 같다.
- 같은 게임을 진행해도 같은 티어의 유저들이 매칭되는 것이 아니다보니 이렇게 집계된듯?
- 그래서 같은 실제 게임이 여러 티어 파일에 중복 포함되어 있는 것으로 보임.

In [159]:
# 같은 gameId가 여러 티어에 걸쳐 있는 경우 몇개..
game_tier_counts = merged_df.groupby("gameId")["tier"].nunique()
game_tier_counts.value_counts().sort_index()

tier
1    49962
2       19
Name: count, dtype: int64

In [160]:
merged_df.duplicated(subset=[
    "gameId", "gameDuration", "level", "lastRound", "Ranked",
    "ingameDuration", "combination", "champion"
]).sum()

np.int64(220)

- tier만 다르거나, 혹은 아예 완전히 같은 게임 행이 총 220개 중복된다.
- 위에서 19게임이었으니까 x8명 하면 152행 => tier만 다른 중복
- 나머지는 같은 파일안에서 완전 중복 or 일부 게임이 16행으로 중복일듯?

# 전처리 아이디어
1. tier 제외 기준으로 중복 제거
2. gameId별로 정확히 8명인지?
3. Ranked가 정확히 1~8인지
4. 정상 게임만 최종 분석용으로 사용
5. 시즌2, 시즌3 구분 => 시즌2 삭제..

## combination 컬럼

In [161]:
merged_df['combination'].dtype

<StringDtype(storage='python', na_value=nan)>

In [162]:
#str -> dic로 변경
import ast

merged_df['combination'] = merged_df['combination'].apply(ast.literal_eval)

In [163]:
type(merged_df['combination'].iloc[0])

dict

In [164]:
#전체 시너지 조회
all_traits = set()

for x in merged_df['combination']:
    all_traits.update(x.keys())

all_traits

{'Alchemist',
 'Avatar',
 'Berserker',
 'Blaster',
 'Celestial',
 'Chrono',
 'Crystal',
 'Cybernetic',
 'DarkStar',
 'Demolitionist',
 'Desert',
 'Druid',
 'Electric',
 'Inferno',
 'Infiltrator',
 'Light',
 'Mage',
 'ManaReaver',
 'MechPilot',
 'Mercenary',
 'Metal',
 'Mountain',
 'Mystic',
 'Ocean',
 'Poison',
 'Predator',
 'Protector',
 'Rebel',
 'Set2_Assassin',
 'Set2_Blademaster',
 'Set2_Glacial',
 'Set2_Ranger',
 'Set3_Blademaster',
 'Set3_Brawler',
 'Set3_Celestial',
 'Set3_Mystic',
 'Set3_Sorcerer',
 'Set3_Void',
 'Shadow',
 'Sniper',
 'Soulbound',
 'SpacePirate',
 'StarGuardian',
 'Starship',
 'Summoner',
 'TemplateTrait',
 'Valkyrie',
 'Vanguard',
 'Warden',
 'Wind',
 'Woodland'}

- g선생에게 정리를 부탁하는데 'Metal','TemplateTrait' 두개는 시즌2, 시즌3에도 확인되지 않음

# 'Metal','TemplateTrait' 확인하기

In [180]:
# 1) 해당 값이 들어간 행 보기
mask = merged_df['combination'].apply(lambda x: 'Metal' in x or 'TemplateTrait' in x)
merged_df[mask].head()

,gameId,gameDuration,level,lastRound,Ranked,ingameDuration,combination,champion,tier
0,KR_4247538593,2142.470703,8,35,1,2134.272217,"{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'S...","{'JarvanIV': {'items': [27], 'star': 3}, 'Sona...",Challenger
3,KR_4247538593,2142.470703,8,33,4,1998.146729,"{'Blaster': 1, 'Cybernetic': 1, 'DarkStar': 1,...","{'KaiSa': {'items': [44, 37], 'star': 2}, 'Ann...",Challenger
7,KR_4247538593,2142.470703,8,28,8,1694.417358,"{'Blaster': 4, 'Chrono': 2, 'Cybernetic': 2, '...","{'Malphite': {'items': [], 'star': 2}, 'Blitzc...",Challenger
9,KR_4247595409,2386.106201,9,41,2,2377.768066,"{'Blaster': 4, 'Chrono': 2, 'Cybernetic': 2, '...","{'Malphite': {'items': [], 'star': 1}, 'Blitzc...",Challenger
11,KR_4247595409,2386.106201,6,31,4,1860.638428,"{'DarkStar': 2, 'Infiltrator': 1, 'Protector':...","{'JarvanIV': {'items': [27, 9], 'star': 3}, 'S...",Challenger


In [181]:
# 2) 각각 몇 번 나오는지
from collections import Counter

cnt = Counter()
for combos in merged_df['combination']:
    for trait in combos:
        if trait in ['Metal', 'TemplateTrait']:
            cnt[trait] += 1

print(cnt)

Counter({'TemplateTrait': 65988, 'Metal': 494})


- 'TemplateTrait' 너무많은데..? 

In [182]:
# 3) 어느 티어에서 나오는지
merged_df[mask].groupby('tier').size()

tier
Challenger     14538
Diamond        12537
GrandMaster    14315
Master         13971
Platinum       11121
dtype: int64

## TemplateTrait 확인

- 특정 테이블에서만 나오는게 아니고 전체적으로 있는 것 같음
- g선생에게 물어보니 데이터 생성과정에서 반복적으로 붙은 기본값 = 가짜(or임시) trait / 찌꺼기일 가능성이 높다고 함. 

In [206]:
# 1. 첫 번째 사례 전체 보기
# 'combination' 컬럼에서 'TemplateTrait'가 들어있는 행만 찾고,
# 그 중 첫 번째 행의 조합 정보를 직접 출력 확인
row = merged_df[merged_df['combination'].apply(lambda x: 'TemplateTrait' in x)]['combination'].iloc[0]
print(row)

{'DarkStar': 2, 'Protector': 4, 'Rebel': 1, 'Set3_Celestial': 3, 'Set3_Mystic': 4, 'StarGuardian': 2, 'TemplateTrait': 1}


- loc[0] , loc[1] 순서대로 확인 해 봤는데 둘다 'TemplateTrait': 1 로 확인됨.

In [200]:
# 2. TemplateTrait 가 없는 행(NaN)과 있는 행(1.0) 조회
merged_df['combination'].apply(
    lambda x: x.get('TemplateTrait') if 'TemplateTrait' in x else None
).value_counts(dropna=False)

combination
NaN    334010
1.0     65988
Name: count, dtype: int64

In [207]:
# 3. TemplateTrait이 포함된 첫번째 행이 다른 정상 trait들과 같이 어떻게 섞이는지 보기
for k, v in row.items():
    print(f"{k}: {v}")

DarkStar: 2
Protector: 4
Rebel: 1
Set3_Celestial: 3
Set3_Mystic: 4
StarGuardian: 2
TemplateTrait: 1


- 'TemplateTrait' 는 단독으로 사용되진 않고
- 정상적인 시즌3 시너지들과 함께 포함되어있는 것 같다.
- 핵심 시너지라기보다 정상 조합 정보에 부가적으로 붙어 있는 불필요한 값으로보인다.

## Metal 확인

In [218]:
# 1. 첫 번째 사례 전체 보기
# 'combination' 컬럼에서 'Metal'이 들어있는 행만 찾고,
# 그 중 첫 번째 행의 조합 정보를 직접 출력 확인
row_metal = merged_df[merged_df['combination'].apply(lambda x: 'Metal' in x)]['combination'].iloc[0]
print(row_metal)

{'Berserker': 1, 'Electric': 2, 'Light': 1, 'Metal': 1, 'Ocean': 1, 'Poison': 1, 'Predator': 3, 'Set2_Glacial': 4, 'Warden': 4}


- iloc 바꾸면서 찍어봣는데 set_2가 많이보임
- set2에만 있는 걸까?

In [224]:
# 시즌3 trait 목록
season3_traits = {
    "Celestial", "Chrono", "Cybernetic", "DarkStar", "Rebel", "MechPilot",
    "SpacePirate", "StarGuardian", "Starship", "Valkyrie", "Void", "Set3_Void",
    "ManaReaver", "Mercenary", "Protector", "Sniper", "Sorcerer", "Set3_Sorcerer",
    "Infiltrator", "Blaster", "Demolitionist", "Brawler", "Set3_Brawler",
    "Vanguard", "Set3_Mystic", "Set3_Blademaster", "Set3_Celestial"
}

In [225]:
# Metal이 포함된 조합에 시즌3 시너지가 함께 있는지 확인
metal_df = merged_df[merged_df['combination'].apply(lambda x: 'Metal' in x)].copy()

metal_df['has_set3'] = metal_df['combination'].apply(
    lambda d: any(trait in season3_traits for trait in d.keys())
)

#시즌3 시너지 포함 여부 개수
metal_df['has_set3'].value_counts()

has_set3
False    475
True      19
Name: count, dtype: int64

- Metal은 대부분 시즌2쪽 조합에 등장하는 듯?

In [214]:
# 2. Metal 가 없는 행(NaN)과 있는 행 조회
merged_df['combination'].apply(
    lambda x: x.get('Metal') if 'Metal' in x else None
).value_counts(dropna=False)

combination
NaN    399504
2.0       264
1.0       202
4.0        19
3.0         9
Name: count, dtype: int64

In [219]:
# 3. Metal 포함된 첫번째 행이 다른 정상 trait들과 같이 어떻게 섞이는지 보기
for k, v in row_metal.items():
    print(f"{k}: {v}")

Berserker: 1
Electric: 2
Light: 1
Metal: 1
Ocean: 1
Poison: 1
Predator: 3
Set2_Glacial: 4
Warden: 4


- 그럼 시즌2를 날린다면 19건만 있을거니까
- 오염값? 이라고 특정하고 지워도 되지 않을까?

## 시즌 2... 어케할까?

In [165]:
#준환님이 정리해준 시즌2 시너지목록/ 일부수정(Set2_)
season2_traits = {
    "Steel", "Shadow", "Lunar", "Mountain", "Poison", "Ocean", "Wind", "Set2_Glacial",
    "Light", "Desert", "Woodland", "Electric", "Inferno", "Set2_Blademaster", "Berserker",
    "Druid", "Summoner", "Mystic", "Avatar", "Set2_Assassin", "Alchemist", "Soulbound",
    "Mage", "Set2_Ranger", "Warden", "Predator", "Crystal"
}

In [166]:
# 시즌3 trait 목록
season3_traits = {
    "Celestial", "Chrono", "Cybernetic", "DarkStar", "Rebel", "MechPilot",
    "SpacePirate", "StarGuardian", "Starship", "Valkyrie", "Void", "Set3_Void",
    "ManaReaver", "Mercenary", "Protector", "Sniper", "Sorcerer", "Set3_Sorcerer",
    "Infiltrator", "Blaster", "Demolitionist", "Brawler", "Set3_Brawler",
    "Vanguard", "Set3_Mystic", "Set3_Blademaster", "Set3_Celestial"
}

In [167]:
print("season2 개수:", len(season2_traits))
print("season3 개수:", len(season3_traits))
print("all_traits 개수:", len(all_traits))
print("미분류 개수:", len(all_traits - season2_traits - season3_traits))

season2 개수: 27
season3 개수: 27
all_traits 개수: 51
미분류 개수: 2


In [168]:
sorted(season2_traits - all_traits)

['Lunar', 'Steel']

Lunar,Steel은 시즌 2로 정리 해 주셨지만 실제 우리 데이터에 없는 것 같다.(비인기 시너지?)

## 시즌2 포함된 행개수와 비중

In [169]:
def has_season2_trait(trait_list):
    for trait in trait_list:
        if trait in season2_traits:
            return True
    return False

season2_mask = merged_df['combination'].apply(has_season2_trait)

count = season2_mask.sum()
ratio = season2_mask.mean() * 100
total = len(merged_df)

print(f"전체 행 수: {total}")
print(f"시즌2 포함 행 개수: {count}")
print(f"시즌2 포함 비중: {ratio:.2f}%")

전체 행 수: 399998
시즌2 포함 행 개수: 3230
시즌2 포함 비중: 0.81%


# 전처리 아이디어
1. 시즌2 데이터는 삭제해도 될 것 같다.

## df_champion 데이터 확인

In [179]:
print(df_champion.shape)
print(df_champion.columns)
print(df_champion.info())
df_champion.head()

(52, 12)
Index(['name', 'cost', 'health', 'defense', 'attack', 'attack_range',
       'speed_of_attack', 'dps', 'skill_name', 'skill_cost', 'origin',
       'class'],
      dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             52 non-null     str    
 1   cost             52 non-null     int64  
 2   health           52 non-null     int64  
 3   defense          52 non-null     int64  
 4   attack           52 non-null     int64  
 5   attack_range     52 non-null     int64  
 6   speed_of_attack  52 non-null     float64
 7   dps              52 non-null     int64  
 8   skill_name       52 non-null     str    
 9   skill_cost       52 non-null     str    
 10  origin           52 non-null     str    
 11  class            52 non-null     str    
dtypes: float64(1), int64(6), str(5)
memory usage: 5.0 KB
None


,name,cost,health,defense,attack,attack_range,speed_of_attack,dps,skill_name,skill_cost,origin,class
0,gangplank,5,1000,30,60,1,1.00,60,gangplank_orbitalstrike,100/175,Space Pirate,"['Mercenary', 'Demolitionist']"
1,graves,1,650,35,55,1,0.55,30,graves_smokegrenade,50/80,Space Pirate,['Blaster']
2,neeko,3,800,35,50,2,0.65,33,neeko_popblossom,75/150,Star Guardian,['Protector']
3,darius,2,750,35,60,1,0.65,39,darius_spacepirateguillotine,0/60,Space Pirate,['Mana-Reaver']
4,rakan,2,600,35,45,2,0.70,32,rakan_grandentrance,50/100,Celestial,['Protector']


In [171]:
#champion 테이블 중복확인
df_champion["name"].duplicated().sum()

np.int64(0)

## df_item 데이터 확인

In [175]:
print(df_item.shape)
print(df_item.columns)
print(df_item.info())
df_item.head()

(54, 2)
Index(['id', 'name'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      54 non-null     int64
 1   name    54 non-null     str  
dtypes: int64(1), str(1)
memory usage: 996.0 bytes
None


,id,name
0,1,B.F. Sword
1,2,Recurve Bow
2,3,Needlessly Large Rod
3,4,Tear of the Goddess
4,5,Chain Vest


In [173]:
#df_item 중복 확인
df_item.duplicated().sum()

np.int64(0)